# Anchor-projection features from review embeddings

Raw 384-dim embeddings (and their PCA compression) *worsened* both price and
rating models — too wide/noisy for XGBoost on ~113k rows. This notebook takes
the **interpretable** route instead: project each review embedding onto a small
set of named wine concepts and keep only those similarity scores as features.

Method (**multi-anchor**):
1. Each concept in `ANCHORS_MULTI` is described by several anchor sentences.
2. Encode the anchors with the *same* model as `06` (`all-MiniLM-L6-v2`),
   average them into one L2-normalised **concept centroid** (averaging several
   phrasings is more robust than a single anchor sentence).
3. Cosine-similarity of every (cached) review embedding against each centroid
   → one `anchor_<concept>` feature per concept.

Output: `features_embeddings_anchored.parquet`, keyed by `wine_id`, joinable to
`features_basic` like the other feature modules. These are the semantic analog
of the lexical `features_keywords` — same idea, but meaning-based not word-match.

In [ ]:
import numpy as np
import pandas as pd
import itables
from itables import show
from sentence_transformers import SentenceTransformer

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

EMBEDDINGS_PATH = r"..\..\features\features_embeddings.parquet"   # from 06_nlp_embeddings
ANCHORED_PATH   = r"..\..\features\features_embeddings_anchored.parquet"
SILVER_PATH     = r"..\..\.data\wine_reviews_silver.parquet"   # for the sanity check only
MODEL_NAME      = "all-MiniLM-L6-v2"

emb = pd.read_parquet(EMBEDDINGS_PATH)
emb_cols = [c for c in emb.columns if c.startswith("emb_")]
print(f"embeddings: {emb.shape}  ({len(emb_cols)} dims)")
assert emb["wine_id"].is_unique, "wine_id must be unique"

c:\Users\Mateusz\PycharmProjects\wines\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


embeddings: (135192, 385)  (384 dims)


## Anchor concepts

Each concept is defined by a few example sentences. Edit freely — everything
below is driven by this dict.

In [2]:
ANCHORS_MULTI = {
    "dry": [
        "This wine is bone dry.",
        "A dry style with no residual sugar.",
        "Crisp and dry on the finish.",
    ],
    "acidic": [
        "Bright, zesty acidity.",
        "Crisp acidity drives the palate.",
        "Vibrant, mouthwatering acidity.",
    ],
    "sweet": [
        "This wine is sweet and lush.",
        "Noticeable residual sugar on the palate.",
        "A rich, sugary sweetness.",
    ],
    "alcohol": [
        "A hot, boozy finish.",
        "Noticeable alcoholic warmth.",
        "High alcohol gives a burning sensation.",
    ],

    "citrus_fruits": [
        "Aromas of lemon zest.",
        "Notes of orange peel.",
        "Bright grapefruit and citrus flavors.",
    ],
    "green_fruits": [
        "Notes of green apple.",
        "Crisp pear flavors.",
        "Orchard fruit aromas of apple and pear.",
    ],
    "red_fruits": [
        "Flavors of ripe strawberry.",
        "Bright cherry notes.",
        "Juicy raspberry fruit.",
    ],
    "black_fruits": [
        "Rich blackberry aromas.",
        "Dark notes of black cherry.",
        "Concentrated black currant fruit.",
    ],
    "tropical_fruits": [
        "Exotic aromas of pineapple.",
        "Ripe melon notes.",
        "Notes of banana and guava.",
    ],
    "stone_fruits": [
        "Juicy peach flavors.",
        "Aromas of ripe apricot.",
        "Notes of nectarine.",
    ],

    "oak": [
        "Notes of oak on the finish.",
        "Toasted wood aromas.",
        "Aged in oak barrels, with woody notes.",
    ],
    "coconut": [
        "Aromas of coconut.",
        "Notes of toasted coconut.",
        "A coconut-like creaminess from oak aging.",
    ],
    "vanilla": [
        "Sweet vanilla notes.",
        "Aromas of vanilla bean.",
        "Vanilla and baking spice from oak.",
    ],
    "smoke": [
        "Smoky aromas.",
        "Notes of charred wood.",
        "A campfire-like smokiness.",
    ],

    "tannic": [
        "Firm, grippy tannins.",
        "Structured, drying tannins.",
        "Robust tannic backbone.",
    ],
    "body": [
        "A full-bodied wine.",
        "Rich, weighty texture.",
        "Light-bodied and delicate on the palate.",
    ],

    "flowers": [
        "Floral aromas of violet.",
        "Notes of rose petal.",
        "Delicate jasmine and blossom aromas.",
    ],
    "herbs": [
        "Herbal notes of thyme.",
        "Aromas of fresh sage.",
        "Notes of fresh-cut grass.",
    ],
    "spicy": [
        "Peppery spice notes.",
        "Aromas of clove.",
        "Notes of black pepper.",
    ],
    "vegetables": [
        "Green, vegetal notes.",
        "Aromas of bell pepper.",
        "Notes of tomato leaf.",
    ],
    "earth": [
        "Earthy mineral notes.",
        "Aromas of wet stone.",
        "Notes of forest floor.",
    ],
    "leather": [
        "Notes of leather.",
        "Aromas of game and leather.",
        "A leathery, savory complexity.",
    ],
}

concepts = list(ANCHORS_MULTI)
print(f"{len(concepts)} concepts, "
      f"{sum(len(v) for v in ANCHORS_MULTI.values())} anchor sentences")

22 concepts, 66 anchor sentences


## Build concept centroids

Encode every anchor sentence with the same model as `06`, then per concept
average the (unit) anchor vectors and re-normalise → one direction per concept.

In [3]:
model = SentenceTransformer(MODEL_NAME)
print(f"model dim: {model.get_sentence_embedding_dimension()}")

centroids = []
for c in concepts:
    a = model.encode(ANCHORS_MULTI[c], convert_to_numpy=True, normalize_embeddings=True)
    v = a.mean(axis=0)
    v = v / (np.linalg.norm(v) + 1e-12)
    centroids.append(v)

A = np.vstack(centroids).astype(np.float32)         # (n_concepts, dim)
assert A.shape[1] == len(emb_cols), "anchor dim != embedding dim"
print(f"centroid matrix A: {A.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5367.83it/s]
C:\Users\Mateusz\AppData\Local\Temp\ipykernel_24256\3665375727.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model dim: {model.get_sentence_embedding_dimension()}")


model dim: 384
centroid matrix A: (22, 384)


## Project reviews onto the anchors

L2-normalise the cached review embeddings, then a single matrix multiply gives
the cosine similarity of each review to each concept centroid.

In [4]:
E = emb[emb_cols].to_numpy(np.float32)
E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)

sims = (E @ A.T).astype(np.float32)                 # (N, n_concepts) cosine in [-1, 1]
anchor_cols = [f"anchor_{c}" for c in concepts]
anchor_df = pd.DataFrame(sims, columns=anchor_cols, index=emb.index)
print(f"anchor features: {anchor_df.shape}")
anchor_df.describe().T[["mean", "std", "min", "max"]].round(3)

anchor features: (135192, 22)


,mean,std,min,max
anchor_dry,0.380,0.060,0.008,0.686
anchor_acidic,0.400,0.074,-0.009,0.713
anchor_sweet,0.511,0.067,-0.036,0.745
anchor_alcohol,0.328,0.062,-0.023,0.606
anchor_citrus_fruits,0.482,0.082,-0.021,0.779
anchor_green_fruits,0.459,0.091,-0.016,0.835
anchor_red_fruits,0.469,0.081,0.009,0.793
anchor_black_fruits,0.490,0.077,0.015,0.776
anchor_tropical_fruits,0.454,0.084,0.001,0.767
anchor_stone_fruits,0.486,0.085,-0.025,0.820


## Sanity check — top concepts per review

For a few sample wines, the highest-scoring concepts should match what the
review actually says. Pulls the review text from Silver (join on `wine_id`).

In [5]:
silver = pd.read_parquet(SILVER_PATH)[["wine_id", "review"]]
sample = anchor_df.join(emb["wine_id"]).sample(6, random_state=7)
sample = sample.merge(silver, on="wine_id", how="left")

for _, row in sample.iterrows():
    top = row[anchor_cols].astype(float).sort_values(ascending=False).head(4)
    tags = ", ".join(f"{c.replace('anchor_', '')} {v:.2f}" for c, v in top.items())
    print(f"- {str(row['review'])[:170]}")
    print(f"    -> {tags}\n")

- Lane Tanner, the first independent female winemaker in Santa Barbara County history, is applying her Pinot Noir acumen to Grenache, with great results. This bright and fr
    -> black_fruits 0.39, sweet 0.39, red_fruits 0.37, citrus_fruits 0.36

- This wine is from the Manilla vineyard, which sits on volcanic soils and is situated in a natural clos, or protected area that results in more humidity and climactic stab
    -> body 0.47, sweet 0.46, citrus_fruits 0.42, red_fruits 0.41

- The aromatic combination of sweet petunias, blackberry jam and hoisin sauce is a winning one. Muscular tannins support bowls of blackberry and cherry flavors, along with 
    -> stone_fruits 0.58, black_fruits 0.52, tannic 0.51, tropical_fruits 0.51

- Aromas of pepper, wet slate tart blackberries and wild fennel swirl together to create a refreshing, invigorating nose. The palate layers more blackberries over black lic
    -> black_fruits 0.58, spicy 0.55, vanilla 0.53, stone_fruits 0.50

- The nose glow

## Save

`wine_id` + 22 `anchor_*` cosine features. Joins to `features_basic` on
`wine_id`; use in `models/01_models_retail.ipynb` / `02_models_rating.ipynb`
as a `base + anchors` model alongside `base + kw`.

In [6]:
out = pd.concat([emb[["wine_id"]], anchor_df], axis=1)
out.to_parquet(ANCHORED_PATH, index=False)
size_mb = __import__("os").path.getsize(ANCHORED_PATH) / 1e6
print(f"Saved {out.shape[0]:,} rows x {out.shape[1]} cols ({size_mb:.1f} MB) -> {ANCHORED_PATH}")
out.head()

Saved 135,192 rows x 23 cols (18.7 MB) -> ..\..\.data\features_embeddings_anchored.parquet


,wine_id,anchor_dry,anchor_acidic,anchor_sweet,anchor_alcohol,anchor_citrus_fruits,anchor_green_fruits,anchor_red_fruits,anchor_black_fruits,anchor_tropical_fruits,...,anchor_vanilla,anchor_smoke,anchor_tannic,anchor_body,anchor_flowers,anchor_herbs,anchor_spicy,anchor_vegetables,anchor_earth,anchor_leather
0,0,0.504971,0.493313,0.635010,0.388399,0.506058,0.467482,0.500822,0.525272,0.462642,...,0.546279,0.366973,0.383797,0.588324,0.459899,0.495154,0.460979,0.374130,0.414846,0.422472
1,1,0.270563,0.445601,0.554325,0.296260,0.490587,0.513949,0.612728,0.614764,0.512592,...,0.539772,0.303429,0.278267,0.457505,0.492797,0.440917,0.527762,0.465447,0.311941,0.286801
2,2,0.293036,0.368024,0.429985,0.236756,0.513019,0.438555,0.521742,0.508451,0.552401,...,0.508207,0.334452,0.304497,0.360258,0.427135,0.449677,0.522400,0.443967,0.279937,0.256382
3,3,0.419526,0.337207,0.594985,0.409876,0.480047,0.414460,0.503209,0.549152,0.411739,...,0.412499,0.249928,0.293616,0.536887,0.394816,0.357397,0.391884,0.303063,0.272610,0.357901
4,4,0.393198,0.369924,0.619882,0.445907,0.451717,0.424325,0.477352,0.445184,0.404133,...,0.397398,0.296955,0.287521,0.546151,0.403179,0.323128,0.391317,0.358649,0.302048,0.325512
